In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import time

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import r2_score

from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor, GradientBoostingRegressor, RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Reproducibility (best effort; GPU can still introduce tiny nondeterminism)
import os, random
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

import warnings
warnings.filterwarnings('ignore')

In [2]:
# Load data 

df = pd.read_csv("energy_dataset_clean.csv", parse_dates=["time"], index_col="time")

# Convert directly to timezone-aware  
df.index = pd.to_datetime(df.index, utc=True).tz_convert('Europe/Paris')

# Optional: set hourly frequency  
df = df.asfreq('h')

df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 35064 entries, 2015-01-01 00:00:00+01:00 to 2018-12-31 23:00:00+01:00
Freq: h
Data columns (total 20 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   gen_biomass                           35064 non-null  float64
 1   gen_fossil_brown_coal_lignite         35064 non-null  float64
 2   gen_fossil_gas                        35064 non-null  float64
 3   gen_fossil_hard_coal                  35064 non-null  float64
 4   gen_fossil_oil                        35064 non-null  float64
 5   gen_hydro_pumped_storage_consumption  35064 non-null  float64
 6   gen_hydro_run-of-river_and_poundage   35064 non-null  float64
 7   gen_hydro_water_reservoir             35064 non-null  float64
 8   gen_nuclear                           35064 non-null  float64
 9   gen_other                             35064 non-null  float64
 10  gen_other_renewable        

### Time series sliding window 

In [3]:
# Sliding-window (univariate, direct multi-step)
# Lag convention: lag1  = most recent past observation (t−1);
#                 lag24 = oldest past value (t−24)

def slideWindow_univariate(y, window_in, horizon=1, name=None):
    
    series = y.values
    idx = y.index  # keep the original DatetimeIndex
    n = len(series)
    n_samples = n - window_in - horizon + 1
    
    if n_samples <= 0:
        raise ValueError("Not enough data for given window_in and horizon")
    
    # Pre-allocate list of rows (fast + clean)
    rows = [] 
    row_index = []  # timestamps for each row (we’ll use t+1 timestamp)
    for i in range(n_samples):
        # Input window: past values
        window = series[i : i + window_in][::-1]  # lag1 = most recent
        input_cols = [f"{name}_lag{t+1}" for t in range(window_in)]
        
        # Output: future values
        target = series[i + window_in : i + window_in + horizon]
        target_cols = [f"{name}_t+{t+1}" for t in range(horizon)]
        
        # Combine
        row = np.concatenate([window, target])
        cols = input_cols + target_cols
        rows.append(pd.Series(row, index=cols))

        # The timestamp associated with this sample = time of t+1
        row_index.append(idx[i + window_in])
    
    return pd.DataFrame(rows, index=pd.DatetimeIndex(row_index))


### Evaluation function

sklearn’s metrics are not ideal for multi-output, horizon-aware time-series evaluation. In mean_absolute_error(y_true, y_pred) if y_true and y_pred are shape (n_samples, 24), sklearn averages over outputs by default and we do not know whether it averaged per horizon then across horizons or flattened everything; and we cannot easily extract per-horizon errors.
So, we will define all metrics explicitly to have full control, clarity, and reproducibility.

The **evaluate_multioutput** function is designed for models that explicitly predict a vector:
(y<sub>t+1</sub>,y<sub>t+2</sub>,...,y<sub>t+24</sub>)
in a single model call.

The function in addition to aggregate error metrics, also computes horizon-specific errors to permit to analyze how forecasting performance evolves with increasing prediction lead time, providing a more detailed assessment of model robustness for day-ahead electricity price forecasting.

In [4]:
def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))
    
def mape(y_true, y_pred, eps=1e-6):
    denom = np.clip(np.abs(y_true), eps, None)
    return np.mean(np.abs((y_true - y_pred) / denom)) * 100

def r2(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true, axis=0)) ** 2)
    if np.isclose(ss_tot, 0.0):
        return 0.0
    return 1.0 - (ss_res / ss_tot)

def seasonal_naive_scale(y_train_full, seasonality=24):
    """
    Compute MASE denominator using seasonal naive forecast errors
    on the in-sample training series.
    y_train_full: 1D array-like
    """
    y_train_full = np.asarray(y_train_full).astype(float)
    if len(y_train_full) <= seasonality:
        raise ValueError("Training series too short for MASE seasonality.")
    
    return np.mean(np.abs(y_train_full[seasonality:] - y_train_full[:-seasonality]))

def mase(y_true, y_pred, y_train_full):

    mase_denom = seasonal_naive_scale(y_train_full, seasonality=24)
    
    return np.mean(np.abs(y_true - y_pred)) / mase_denom

def evaluate_multioutput(y_true, y_pred, y_train_full=None):
    """
    y_true, y_pred: shape (samples, horizons)
    Returns aggregated metrics.
    """
    out = {
        "MAE_mean": round(mae(y_true, y_pred),3),
        "RMSE_mean": round(rmse(y_true, y_pred),3),
        "MAPE_mean": round(mape(y_true, y_pred),3),
        "R2_mean": round(r2(y_true, y_pred),3),
    } 
    if y_train_full is not None:
        out["MASE_mean"] = round(mase(y_true, y_pred, y_train_full),3)
        
    return out

In [5]:
# All models + multi-output handling
 
def get_models(random_state=42):
    models = {
        "LinearRegression": LinearRegression(),
        "GradientBoosting": GradientBoostingRegressor(random_state=random_state),
        "RandomForest": RandomForestRegressor(n_estimators=600, random_state=random_state, n_jobs=-1),
        "XGBoost": XGBRegressor(n_estimators=1200,
                                learning_rate=0.05,
                                 max_depth=6,
                                 subsample=0.9,
                                 colsample_bytree=0.9,
                                 reg_lambda=1.0,
                                 random_state=random_state,
                                 n_jobs=-1),
        "LightGBM": LGBMRegressor(n_estimators=2000,
                                  learning_rate=0.03,
                                  num_leaves=63,
                                  subsample=0.9,
                                  colsample_bytree=0.9,
                                  random_state=random_state,
                                  n_jobs=-1,
                                  verbosity=-1),  # Set verbosity to -1 to disable messages
         "Bagging": BaggingRegressor(estimator=DecisionTreeRegressor(random_state=random_state),
                                    n_estimators=200,
                                    random_state=random_state,
                                     n_jobs=-1)
    }
    return models


def make_estimator(model_name: str, base_estimator):
    """
    - Scale only for LinearRegression and KNN (distance / coefficient-sensitive).
    - Wrap XGBoost & LightGBM (and optionally others) with MultiOutputRegressor for 24 outputs.
    """
    needs_scaling = model_name in {"LinearRegression", "KNN"}
    needs_multioutput_wrap = model_name in {"GradientBoosting","XGBoost", "LightGBM"}

    est = base_estimator
    if needs_multioutput_wrap:
        est = MultiOutputRegressor(est, n_jobs=-1)

    if needs_scaling:
        return Pipeline([("scaler", StandardScaler()), ("model", est)])

    return est


# Split data: Sliding expanding-window CV

| Window | Train     | Test |
|--------|-----------|------|
| 1      | 2015      | 2016 |
| 2      | 2015–2016 | 2017 |
| 3      | 2015–2017 | 2018 |

This is a **expanding window cross-validation**, chronologically consistent, with no look-ahead bias, and realistic for operational forecasting.<br>
An expanding-window cross-validation strategy was adopted to preserve the temporal structure of the data and avoid look-ahead bias.


# Test multiple window sizes  
- 24 (one full day) captures daily pattern only
- 48 (yesterday + today) usually good 
- 84 (3.5 days) captures daily + weekend patterns well  
- 168 (7 × 24 = one full week) captures full weekly seasonality + weekend effect
- 192 1 week + 1 day

### Two-stage design to select the best window_size:
**First stage**:
- Train 2015 → validate 2016
- Train 2015–2016 → validate 2017

Pick best window_size by average MAE/RMSE across 2016+2017.<br>
**Second stage** - Final evaluation (one untouched year):
- Train 2015–2017 with the chosen window_size
- Test on 2018 only


For each experiment, models were instantiated using a **fresh clone** of the base estimator to avoid information leakage and ensure independence across cross-validation windows and hyperparameter configurations.<br> 
Evaluation metrics were implemented explicitly to enable stable and horizon-specific assessment of multi-step forecasts, particularly important in the presence of near-zero electricity prices.<br>

In [6]:
def run_windows_for_window_sizes(df, target, horizon, cv_windows, window_sizes, random_state=42):
    
    if target not in df.columns:
        raise ValueError(f"'{target}' not in df.columns")
    if not isinstance(df.index, pd.DatetimeIndex):
        raise ValueError("df must have a DatetimeIndex")

    models = get_models(random_state=random_state)
    
    results = []
    for i, window in enumerate(cv_windows, 1):
        train_years = window["train_years"]
        test_year = window["test_year"]
        
        train_df = df[df['year'].isin(train_years)]
        test_df = df[df['year'] == test_year]
     
        for window in window_sizes:

            train = slideWindow_univariate(train_df[target], window, horizon, target)
            test = slideWindow_univariate(test_df[target], window, horizon, target)
    
            X_train = train.iloc[:,:-horizon]         # All Features, except horizon columns
            y_train = train.iloc[:,-horizon:]         # Target - horizon columns
    
            X_test = test.iloc[:,:-horizon]         
            y_test = test.iloc[:,-horizon:] 

            for model_name, base_model in models.items():
                est = make_estimator(model_name, clone(base_model))   # clone(base_model) is to guarantee a fresh, independent model instance for every experiment.
                est.fit(X_train, y_train)
                y_pred = est.predict(X_test)
    
                metrics = evaluate_multioutput(y_test.to_numpy(), y_pred)
    
                row = {
                    "cv_window": f"{i}",
                    "years": f"{train_years} - {test_year}",
                    "window_size": window,
                    "model": model_name,
                }
                row.update(metrics)
                results.append(row)

    return pd.DataFrame(results)
    

## Define the best window size

In [7]:
# ---- First Stage: tuning windows (NO 2018) ---- 
StartTime = time.time()

df['year'] = df.index.year

cv_windows = [
    {"train_years": [2015], "test_year": 2016},
    {"train_years": [2015, 2016], "test_year": 2017},
]

window_sizes = [48, 84, 168, 192]
horizon = 24
target = "price"

tuning_df = run_windows_for_window_sizes(df, target, horizon, cv_windows, window_sizes, random_state=42)

EndTime = time.time()
print(f"Total time: {round((EndTime-StartTime)/60,1)} min")

Total time: 72.5 min


# RNN: GRU and LSTM

RNN uses sequences, not “tabular lags”

For LSTM/GRU you should feed:<br>
X: shape (n_samples, window_size, n_features)<br>
y: shape (n_samples, horizon)<br>

So the “windowing” function differs from Sliding Window tabular for ML methods.

### Sequence windowing for multivariate input + direct 24-step output 

In [8]:
def make_rnn_sequences(df: pd.DataFrame, feature_cols, target_col: str,
                       window_size: int, horizon: int):
    """
    Builds:
      X: (n_samples, window_size, n_features)
      y: (n_samples, horizon)
      t1_index: DatetimeIndex for the first predicted hour (t+1) of each sample
      tend_index: DatetimeIndex for the last predicted hour (t+horizon) of each sample
    """
    df = df.sort_index()
    X_vals = df[feature_cols].to_numpy(dtype=np.float32)
    y_vals = df[target_col].to_numpy(dtype=np.float32)
    idx = df.index

    n = len(df)
    n_samples = n - window_size - horizon + 1
    if n_samples <= 0:
        raise ValueError("Not enough data for given window_size and horizon")

    X = np.empty((n_samples, window_size, len(feature_cols)), dtype=np.float32)
    y = np.empty((n_samples, horizon), dtype=np.float32)

    t1_list = []
    tend_list = []

    for i in range(n_samples):
        X[i] = X_vals[i : i + window_size, :]
        y[i] = y_vals[i + window_size : i + window_size + horizon]

        t1_list.append(idx[i + window_size])                 # t+1 timestamp
        tend_list.append(idx[i + window_size + horizon - 1])  # t+horizon timestamp

    return X, y, pd.DatetimeIndex(t1_list), pd.DatetimeIndex(tend_list)
    

### Scaling (train-only) for DL

For DL, we must scale inputs (and optionally scale targets). To keep comparisons clean, we can not scale targets; but for training stability, scaling targets often helps. We will scale both, and we inverse-transform predictions.

In [9]:
def fit_transform_scalers(X_train, y_train):
    """
    Fit scalers on train only.
    X scaler is applied feature-wise across all timesteps.
    y scaler is applied across horizons (t+1..t+24).
    """
    n_train, T, F = X_train.shape

    x_scaler = StandardScaler()
    X_train_2d = X_train.reshape(-1, F)
    X_train_scaled_2d = x_scaler.fit_transform(X_train_2d)
    X_train_scaled = X_train_scaled_2d.reshape(n_train, T, F)

    y_scaler = StandardScaler()
    y_train_scaled = y_scaler.fit_transform(y_train)

    return x_scaler, y_scaler, X_train_scaled, y_train_scaled

def transform_with_scalers(X, y, x_scaler, y_scaler):
    n, T, F = X.shape
    X_2d = X.reshape(-1, F)
    X_scaled = x_scaler.transform(X_2d).reshape(n, T, F)
    y_scaled = y_scaler.transform(y)
    
    return X_scaled, y_scaled

### RNN models (GRU and LSTM), direct multi-step output

In [10]:
def build_gru_model(window_size, n_features, horizon,
                    units=64, dropout=0.1, lr=1e-3):
    inp = keras.Input(shape=(window_size, n_features))
    x = layers.GRU(units, dropout=dropout, recurrent_dropout=0.0)(inp)
    x = layers.Dense(64, activation="relu")(x)
    out = layers.Dense(horizon)(x)  # direct multi-step output
    model = keras.Model(inp, out)
    model.compile(optimizer=keras.optimizers.Adam(lr), loss="mse")
    
    return model

def build_lstm_model(window_size, n_features, horizon,
                     units=64, dropout=0.1, lr=1e-3):
    inp = keras.Input(shape=(window_size, n_features))
    x = layers.LSTM(units, dropout=dropout, recurrent_dropout=0.0)(inp)
    x = layers.Dense(64, activation="relu")(x)
    out = layers.Dense(horizon)(x)
    model = keras.Model(inp, out)
    model.compile(optimizer=keras.optimizers.Adam(lr), loss="mse")
    
    return model

### Function to run LSTM/GRU 

In [12]:
def run_rnn_experiment(df_seg, feature_cols, target, window_size, horizon, 
                       train_years=(2015,2016,2017), test_year=2018, 
                       rnn_type="GRU",
                       units=64, dropout=0.1, lr=1e-3,
                       batch_size=256, epochs=50, val_days=30):

    X, y, t1, tend = make_rnn_sequences(df_seg, feature_cols, target, window_size, horizon)

    tz = df_seg.index.tz
    train_end = pd.Timestamp(f"{max(train_years)}-12-31 23:00:00", tz=tz)
    test_start = pd.Timestamp(f"{test_year}-01-01 00:00:00", tz=tz)
    test_end   = pd.Timestamp(f"{test_year}-12-31 23:00:00", tz=tz)

    # Extract the univariate series for the entire training years
    y_train_full = df_seg.loc[:train_end, target].values   # .values → numpy 1D array

    # Masks: train/val from train period; test from test year
    train_mask, val_mask = make_train_val_masks(t1, tend, train_end, val_days=val_days)
    test_mask = (t1 >= test_start) & (tend <= test_end)

    X_train, y_train = X[train_mask], y[train_mask]
    X_val,   y_val   = X[val_mask],   y[val_mask]
    X_test,  y_test  = X[test_mask],  y[test_mask]

    # Scale (fit on train only)
    x_scaler, y_scaler, X_train_s, y_train_s = fit_transform_scalers(X_train, y_train)
    X_val_s,  y_val_s  = transform_with_scalers(X_val, y_val, x_scaler, y_scaler)
    X_test_s, y_test_s = transform_with_scalers(X_test, y_test, x_scaler, y_scaler)

    # Build model
    n_features = X_train.shape[2]
    if rnn_type.upper() == "GRU":
        model = build_gru_model(window_size, n_features, horizon, units=units, dropout=dropout, lr=lr)
    else:
        model = build_lstm_model(window_size, n_features, horizon, units=units, dropout=dropout, lr=lr)

    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", patience=4, factor=0.5, min_lr=1e-5)
    ]

    model.fit(X_train_s, y_train_s,
              validation_data=(X_val_s, y_val_s),
              epochs=epochs,
              batch_size=batch_size,
              verbose=1,
              callbacks=callbacks)

    # Predict and inverse-transform back to original scale
    y_pred_s = model.predict(X_test_s, verbose=0)
    y_pred = y_scaler.inverse_transform(y_pred_s)

    metrics = evaluate_multioutput(y_true = y_test,          # (n_test_windows, 24)
                                   y_pred = y_pred,          # (n_test_windows, 24) 
                                   y_train_full = y_train_full)
    
    return metrics, t1[test_mask], y_train_full, y_pred 

### Training setup (early stopping + validation from train only)

We’ll use a tail validation split from the end of the training period (e.g., last 30 days of 2017). That’s time-consistent and avoids leakage.

In [13]:
def make_train_val_masks(t1, tend, train_end, val_days=30):
    
    val_start = train_end - pd.Timedelta(days=val_days)
    
    # Train samples: fully before val_start
    train_mask = tend < val_start
    
    # Val samples: within [val_start, train_end]
    val_mask = (t1 >= val_start) & (tend <= train_end)
    
    return train_mask, val_mask
    

## Test multiple window sizes 

In [14]:
StartTime = time.time()

window_sizes = [24, 48, 84, 168, 192]
tuning_windows = [
    {"train_years": [2015], "test_year": 2016},
    {"train_years": [2015, 2016], "test_year": 2017},
]

feature_cols = ["price"]   # only price
target = "price"
horizon = 24

tune_rows = []

df_seg_full = df.sort_index().copy()  #full dataframe with DatetimeIndex

for rnn_type in ["GRU", "LSTM"]:
    for w in window_sizes:
        rmse_list, mae_list = [], []
        for win in tuning_windows:
            m,_,_,_ = run_rnn_experiment(
                df_seg=df_seg_full,
                feature_cols=feature_cols,
                target=target,
                window_size=w,
                horizon=horizon,
                train_years=win["train_years"],
                test_year=win["test_year"],
                rnn_type=rnn_type,
            )
            rmse_list.append(m["RMSE_mean"])
            mae_list.append(m["MAE_mean"])

        tune_rows.append({
            "model": rnn_type,
            "window_size": w,
            "RMSE_mean": float(np.mean(rmse_list)),
            "RMSE_std": float(np.std(rmse_list))
        })

tune_df = pd.DataFrame(tune_rows).sort_values(["model", "RMSE_mean"])

EndTime = time.time()
print(f"Total time: {round((EndTime-StartTime)/60,1)} min")

Epoch 1/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step - loss: 0.9031 - val_loss: 0.8342 - learning_rate: 0.0010
Epoch 2/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - loss: 0.7908 - val_loss: 0.7781 - learning_rate: 0.0010
Epoch 3/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 0.7499 - val_loss: 0.6958 - learning_rate: 0.0010
Epoch 4/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 0.6975 - val_loss: 0.6207 - learning_rate: 0.0010
Epoch 5/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 0.6597 - val_loss: 0.5664 - learning_rate: 0.0010
Epoch 6/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 0.6195 - val_loss: 0.5162 - learning_rate: 0.0010
Epoch 7/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 0.5965 - val_loss: 0.4825 - learning_rate: 0.0010
Epoch 8/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 0.5731 - val_loss: 0.4641 - learning_rate: 0.0010
Epoch 9/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 0.5627 - val_loss: 0.4509 - learning_rate: 0.0010
Epoch 10/50
32/32 ━

### Best window size per model:

In [15]:
# Concatenate both DataFrames into a single DataFrame       
best_ws_per_model = (
    tuning_df
    .groupby(['model', 'window_size'], as_index=False)
    .agg(RMSE_mean=('RMSE_mean', 'mean'),
         RMSE_std=('RMSE_mean', 'std'))
    .sort_values(['RMSE_mean','model'])
)

# Concatenate both DataFrames into a single DataFrame       
best_ws_per_model = pd.concat([best_ws_per_model, tune_df], ignore_index=True)
 
selected_ws = (
    best_ws_per_model
    .groupby('model', as_index=False)
    .first()
    .rename(columns={'window_size': 'best_window_size'})
)
selected_ws[['RMSE_mean','RMSE_std']] = selected_ws[['RMSE_mean','RMSE_std']].round(3)
selected_ws.sort_values(['RMSE_mean'])

,model,best_window_size,RMSE_mean,RMSE_std
5,LinearRegression,168,8.928,0.940
2,GradientBoosting,168,9.534,0.601
1,GRU,48,9.964,0.830
4,LightGBM,168,10.073,1.604
6,RandomForest,168,10.137,0.961
0,Bagging,192,10.172,1.030
7,XGBoost,168,10.178,1.508
3,LSTM,84,10.333,1.136


### A single window size for all algorithms (often preferred for fairness):

In [16]:
global_ws = (
    best_ws_per_model
    .groupby('window_size', as_index=False)
    .agg(RMSE_mean=('RMSE_mean', 'mean'),
         RMSE_std=('RMSE_std', 'std'))
    .sort_values('RMSE_mean')
)
print(global_ws)

best_window_size = int(global_ws.iloc[0]['window_size'])
print(f"\nBest window size: {best_window_size}")

   window_size  RMSE_mean  RMSE_std
3          168   9.977625  0.341885
4          192   9.993937  0.320195
2           84  10.296375  0.463272
1           48  10.336500  0.572062
0           24  10.524500  0.151321

Best window size: 168


**Conclusion**: Window size = **168 hours** offers the best predictive performance (RMSE_avg=9.97), (RMSE_std=0.34)

For 24-step-ahead forecasting, the input window size was selected using expanding-window validation over 2016 and 2017. Since the majority of models, including the the majority of models (Linear Regression and Gradient Boosting, GB, RF, LGBM, XGB), achieved their lowest validation error with a 168-hour window, this value was adopted for all models to ensure a fair comparison.

168 = one week, which captures:<br>
daily seasonality (24h)<br>
weekly seasonality (168h)<br>

# 24-step-ahead forecasting over the whole data<br> (train years: 2015-2017; test year: 2018)
### Strategy: Direct multi-step (24 outputs) using MultiOutputRegressor
For multi-step forecasting, a **direct strategy** was employed, where a single multi-output regression model predicts the 24 future hourly prices simultaneously, thus avoiding the accumulation of errors typical of recursive approaches.


In [17]:
def run_expanding_cv_direct_tensor(X_train, y_train, y_train_orig, X_test, y_test, exogs, random_state=42):
    """
    Train all models, evaluate them, and store predictions in a 3D tensor:
        (samples, horizons, models)

    Returns
    -------
    results_df : pd.DataFrame
        Table with evaluation metrics for each model.
    pred_tensor : np.ndarray
        3D tensor of shape (n_samples, n_horizons, n_models)
    model_names : list
        Names of models in the same order as pred_tensor[:, :, i]
    y_true_array : np.ndarray
        Ground truth values of shape (n_samples, n_horizons)
    """
    models = get_models(random_state=random_state)

    results = []
    model_names = []
    preds_list = []

    # Ground truth as numpy
    y_true_array = y_test.to_numpy() if isinstance(y_test, (pd.DataFrame, pd.Series)) else np.asarray(y_test)

    for model_name, base_model in models.items():
        est = make_estimator(model_name, clone(base_model))  # fresh independent instance
        est.fit(X_train, y_train)

        y_pred = est.predict(X_test)
        y_pred = np.asarray(y_pred)

        # Safety check: force 2D shape if model returns 1D unexpectedly
        if y_pred.ndim == 1:
            y_pred = y_pred.reshape(-1, 1)

        metrics = evaluate_multioutput(y_test,y_pred,y_train_orig)

        row = {
            "exogs": exogs,
            "model": model_name
        }
        row.update(metrics)
        results.append(row)

        preds_list.append(y_pred)
        model_names.append(model_name)

    # Stack predictions into tensor: list of (samples, horizons) -> (models, samples, horizons)
    pred_tensor = np.stack(preds_list, axis=0)

    # Reorder to (samples, horizons, models)
    pred_tensor = np.transpose(pred_tensor, (1, 2, 0))

    results_df = pd.DataFrame(results)

    return results_df, pred_tensor, model_names, y_true_array
    

### First option: only price (without exogenous variables)

In [18]:
StartTime = time.time()

best_window_size = 168
target = "price"
horizon = 24

df['year'] = df.index.year

train_years = [2015, 2016, 2017]
test_year = 2018

train_df = df[df['year'].isin(train_years)]
test_df = df[df['year'] == test_year]

y_train_prev = train_df['price']   #y_train complete, ordered hourly training series (before windowing).
    
train = slideWindow_univariate(train_df[target], best_window_size, horizon, target)
test = slideWindow_univariate(test_df[target], best_window_size, horizon, target)

X_train = train.iloc[:,:-horizon]         # All Features, except horizon columns
y_train = train.iloc[:,-horizon:]         # Target - horizon columns
    
X_test = test.iloc[:,:-horizon]         
y_test = test.iloc[:,-horizon:] 

df_price, pred_tensor_onlyprice, model_names, y_trueprice = run_expanding_cv_direct_tensor(X_train, 
                                                                                           y_train, 
                                                                                           y_train_prev, 
                                                                                           X_test, y_test, 'onlyprice', random_state=42)

# Concatenate all DataFrames into a single DataFrame
result_df = pd.DataFrame()
result_df = pd.concat([df_price], ignore_index=True) 

EndTime = time.time()
print(f"Total time: {round((EndTime-StartTime)/60,1)} min")

Total time: 19.2 min


In [19]:
# Calculate the mean RMSE_mean for each model
mean_rmse = result_df.groupby('model')['RMSE_mean'].mean().reset_index()
mean_rmse = mean_rmse.rename(columns={'RMSE_mean': 'OrderRMSE'})

# Merge back with the original DataFrame
merged_df = result_df.merge(mean_rmse, on='model')

# Sort using the OrderRMSE column, with additional columns
sorted_summary = merged_df.sort_values(by='OrderRMSE')[['exogs', 'model', 'MAE_mean', 'RMSE_mean', 'MAPE_mean', 'MASE_mean']]

print(sorted_summary)

       exogs             model  MAE_mean  RMSE_mean  MAPE_mean  MASE_mean
0  onlyprice  LinearRegression     5.724      8.067     16.449      0.722
1  onlyprice  GradientBoosting     5.764      8.247     17.151      0.727
4  onlyprice          LightGBM     5.895      8.465     17.498      0.744
3  onlyprice           XGBoost     5.962      8.520     17.542      0.752
2  onlyprice      RandomForest     6.356      8.684     18.251      0.802
5  onlyprice           Bagging     6.371      8.722     18.270      0.804


### Sliding Window with Static Exogenous Features 

In [20]:
best_window_size = 168

def slideWindow_static_exog(y, X_static, window_in, horizon=1, target_name='price', exog_prefix='exg'):
    """
    Returns a clean DataFrame with:
    - Lagged target (price_lag1, price_lag2, ..., price_lag84)
    - Static exogenous variables at the END of the window (most common choice)
    - Target(s): price_t+1, price_t+2, ...
    """
    y = pd.Series(y, name=target_name)
    idx = y.index
    n = len(y)
    n_samples = n - window_in - horizon + 1
    if n_samples <= 0:
        raise ValueError("Not enough data")
    
    # Pre-allocate lists for speed
    rows = []    
    row_index = []
    for i in range(n_samples):
        # 1. Lagged target
        lag_start = i
        lag_end = i + window_in
        lagged = y.iloc[lag_start:lag_end].values
        lag_cols = [f'{target_name}_lag{t+1}' for t in range(window_in)]
        
        # 2. Static exogenous at the END of the window (time t)
        static_row = X_static.iloc[i + window_in - 1]  # value at time t
        static_cols = [f'{exog_prefix}_{col}' for col in X_static.columns]
        
        # 3. Target (next value(s))
        target_start = i + window_in
        target_end = target_start + horizon
        target_vals = y.iloc[target_start:target_end].values
        target_cols = [f'{target_name}_t+{t+1}' for t in range(horizon)]
        
        # Combine into one row
        row = np.concatenate([lagged, static_row.values, target_vals])
        cols = lag_cols + static_cols + target_cols
        rows.append(pd.Series(row, index=cols))

        row_index.append(idx[i + window_in])
    
    return pd.DataFrame(rows, index=pd.DatetimeIndex(row_index))


### Second option: price and calendar variables

In [21]:
def add_calendar_features(df: pd.DataFrame, tz: str = None, country_holidays: str = "FR"):
    """
    Adds:
      hour_sin, hour_cos
      dow_sin, dow_cos
      month_sin, month_cos
      is_holiday
    Assumes df.index is DatetimeIndex (tz-aware or tz-naive).
    If tz is provided, features are computed in that timezone (recommended for local seasonality).
    """
    if not isinstance(df.index, pd.DatetimeIndex):
        raise ValueError("df.index must be a DatetimeIndex")

    df = df.copy()

    idx = df.index
    if tz is not None:
        idx = idx.tz_convert(tz) if idx.tz is not None else idx.tz_localize(tz)

    hour = idx.hour
    dow = idx.dayofweek          # Monday=0..Sunday=6
    month = idx.month            # 1..12

    df["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    df["hour_cos"] = np.cos(2 * np.pi * hour / 24)

    df["dow_sin"] = np.sin(2 * np.pi * dow / 7)
    df["dow_cos"] = np.cos(2 * np.pi * dow / 7)

    df["month_sin"] = np.sin(2 * np.pi * (month - 1) / 12)
    df["month_cos"] = np.cos(2 * np.pi * (month - 1) / 12)

    # Holidays (country code example: "FR" for France, "ES" for Spain, "PT" for Portugal)
    # If you don't have 'holidays' installed, install it or set is_holiday=0.
    try:
        import holidays
        hol = holidays.country_holidays(country_holidays)
        df["is_holiday"] = idx.normalize().isin(hol).astype(int)
    except Exception:
        df["is_holiday"] = 0

    calendar_vars = ['hour_sin','hour_cos','dow_sin','dow_cos','month_sin','month_cos','is_holiday']

    return df, calendar_vars
    

In [22]:
#Remove all columns except the year and price
dfaux = df[['year','price']].copy()

dfaux, calendar_vars = add_calendar_features(dfaux, tz="Europe/Paris", country_holidays="ES")

n_test = 8760    

train_df = dfaux[:-n_test].copy()
test_df = dfaux[-n_test:].copy()     #last year 2018

exog_train_df = train_df[calendar_vars].copy()    
exog_test_df  = test_df[calendar_vars].copy()

In [23]:
StartTime = time.time()
    
train = slideWindow_static_exog(train_df['price'], exog_train_df, best_window_size, horizon)
test = slideWindow_static_exog(test_df['price'], exog_test_df, best_window_size, horizon)
    
X_train = train.iloc[:,:-horizon]         # All Features, except horizon columns
y_train = train.iloc[:,-horizon:]         # Target - horizon columns
    
X_test = test.iloc[:,:-horizon]         
y_test = test.iloc[:,-horizon:] 

df_calendar, pred_tensor_prccal, model_names, y_trueprccal = run_expanding_cv_direct_tensor(X_train, 
                                                                                            y_train, 
                                                                                            y_train_prev, 
                                                                                            X_test, y_test, 
                                                                                            'price+calendar', random_state=42)

# Concatenate all DataFrames into a single DataFrame       
result_df = pd.concat([result_df, df_calendar], ignore_index=True)

EndTime = time.time()
print(f"Total time: {round((EndTime-StartTime)/60,1)} min")    

Total time: 18.6 min



### Third option: price and some exogenous features 

In [24]:
important_exogs = ['total_load_actual',
                   'gen_wind_onshore',
                   'gen_solar',
                   'gen_fossil_gas',
                   'gen_hydro_pumped_storage_consumption',
                   'humidity',
                   'temperature']

# Create just two useful lags for each important_exogs
for col in important_exogs:
    dfaux[col + '_lag24']  = df[col].shift(24)   # same hour yesterday
    dfaux[col + '_lag168'] = df[col].shift(168)  # same hour last week

# Optional but cheap and helpful: daily average
for col in important_exogs:
    dfaux[col + '_roll24_mean'] = df[col].rolling(24).mean()
    
exog_vars = []
# Add suffixes for important_exogs
for col in important_exogs:
    exog_vars += [col + '_lag24', col + '_lag168', col + '_roll24_mean']

exog_vars = exog_vars + calendar_vars 

print(f"{len(exog_vars)} exogenous variables: {exog_vars}")

exog_vars += ['price']

dfsome = dfaux[exog_vars].copy()

exog_vars = exog_vars[:-1]

n_test = 8760  

train_df = dfsome[:-n_test].copy()
test_df = dfsome[-n_test:].copy()       #last year: 2018

exog_train_df = train_df[exog_vars].copy()    
exog_test_df  = test_df[exog_vars].copy()


28 exogenous variables: ['total_load_actual_lag24', 'total_load_actual_lag168', 'total_load_actual_roll24_mean', 'gen_wind_onshore_lag24', 'gen_wind_onshore_lag168', 'gen_wind_onshore_roll24_mean', 'gen_solar_lag24', 'gen_solar_lag168', 'gen_solar_roll24_mean', 'gen_fossil_gas_lag24', 'gen_fossil_gas_lag168', 'gen_fossil_gas_roll24_mean', 'gen_hydro_pumped_storage_consumption_lag24', 'gen_hydro_pumped_storage_consumption_lag168', 'gen_hydro_pumped_storage_consumption_roll24_mean', 'humidity_lag24', 'humidity_lag168', 'humidity_roll24_mean', 'temperature_lag24', 'temperature_lag168', 'temperature_roll24_mean', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'is_holiday']


In [25]:
# Expanding-window CV runner (direct 24-step)
StartTime = time.time()
        
train = slideWindow_static_exog(train_df['price'], exog_train_df, best_window_size, horizon)
test = slideWindow_static_exog(test_df['price'], exog_test_df, best_window_size, horizon)
    
X_train = train.iloc[:,:-horizon]         # All Features, except horizon columns
y_train = train.iloc[:,-horizon:]         # Target - horizon columns
    
X_test = test.iloc[:,:-horizon]         
y_test = test.iloc[:,-horizon:]

print(f"Shapes: {X_train.shape}, {y_train.shape}, {X_test.shape}, {y_test.shape}")

na_cols = X_train.columns[X_train.isna().any()].tolist()
print("Columns with NaN:", len(na_cols))
print(X_train[na_cols].isna().sum().sort_values(ascending=False).head(20))

# How many rows contain any NaN?
print("Rows with any NaN in X_train:", X_train.isna().any(axis=1).sum())
print("Rows with any NaN in X_test:", X_test.isna().any(axis=1).sum())

# Drop rows with NaNs in X (and also protect against NaNs in y)
mask_train = ~(X_train.isna().any(axis=1) | y_train.isna().any(axis=1))
X_train = X_train.loc[mask_train]
y_train = y_train.loc[mask_train]

mask_test = ~(X_test.isna().any(axis=1) | y_test.isna().any(axis=1))
X_test = X_test.loc[mask_test]
y_test = y_test.loc[mask_test]

print(f"X_train, y_train after cleaning: {X_train.shape}, {y_train.shape}")
print(f"X_test, y_test after cleaning: {X_test.shape}, {y_test.shape}")


df_somexog, pred_tensor_allvars, model_names, y_trueallvars = run_expanding_cv_direct_tensor(X_train, 
                                                                                             y_train, 
                                                                                             y_train_prev, 
                                                                                             X_test, y_test, 
                                                                                             'price+calendar+exogs', random_state=42)

y_test2018 =  y_test.to_numpy() if isinstance(y_test, (pd.DataFrame, pd.Series)) else np.asarray(y_test)

# Concatenate all DataFrames into a single DataFrame
result_df = pd.concat([result_df, df_somexog], ignore_index=True)      

EndTime = time.time()
print(f"Total time: {round((EndTime-StartTime)/60,1)} min")    

Shapes: (26113, 196), (26113, 24), (8569, 196), (8569, 24)
Columns with NaN: 7
exg_total_load_actual_lag168                       1
exg_gen_wind_onshore_lag168                        1
exg_gen_solar_lag168                               1
exg_gen_fossil_gas_lag168                          1
exg_gen_hydro_pumped_storage_consumption_lag168    1
exg_humidity_lag168                                1
exg_temperature_lag168                             1
dtype: int64
Rows with any NaN in X_train: 1
Rows with any NaN in X_test: 0
X_train, y_train after cleaning: (26112, 196), (26112, 24)
X_test, y_test after cleaning: (8569, 196), (8569, 24)
Total time: 21.9 min


In [26]:
pd.Series(model_names).to_csv("model_names.csv", index=False)
np.save("pred_tensor_allvars.npy", pred_tensor_allvars)
np.save("dates_ml.npy",test.index)
np.save("y_train_prev.npy", y_train_prev)
np.save("y_test2018.npy", y_test2018)

## Build a GRU model and a LSTM model 

In [27]:
#Remove all columns except the year and price
dfaux = df[['year','price']].copy()

dfaux, calendar_vars = add_calendar_features(dfaux, tz="Europe/Paris", country_holidays="ES")

#Add exogenous vars
exog_vars = ['total_load_actual','gen_wind_onshore','gen_solar','gen_fossil_gas',
             'gen_hydro_pumped_storage_consumption','humidity','temperature']

for col in exog_vars:
    dfaux[col]  = df[col]  

# Check NaNs (should be 0)
allNas = dfaux.isna().sum().sum()
print("All NaNs: ", allNas )

target = 'price'
impute_cols = calendar_vars + exog_vars + [target]

# Impute missing values if there are NaNs
# Forward-fill is very natural for time series (use last known value)
# Back-fill catches any leading NaNs
if allNas != 0:
    if dfaux[impute_cols].isna().values.any():
        dfaux[impute_cols] = dfaux[impute_cols].ffill().bfill()

# Optional safety check (should now be 0)
print("NaNs after imputation:", dfaux[impute_cols].isna().sum().to_dict())

train_years = [2015, 2016, 2017]
test_year = 2018

train_df = df[df["year"].isin(train_years)].drop(columns=["year"]).copy()
test_df  = df[df["year"] == test_year].drop(columns=["year"]).copy()

#y_train_prev = train_df['price']

print(f"Train: {train_df.shape} Test: {test_df.shape}")

print('Min train date: ', np.min(train_df.index), 'Max train date: ', np.max(train_df.index))
print('Min test date: ', np.min(test_df.index), 'Max test date: ', np.max(test_df.index)) 

feature_sets = {
    "price+calendar+exogs": [target] + calendar_vars + exog_vars
}

All NaNs:  0
NaNs after imputation: {'hour_sin': 0, 'hour_cos': 0, 'dow_sin': 0, 'dow_cos': 0, 'month_sin': 0, 'month_cos': 0, 'is_holiday': 0, 'total_load_actual': 0, 'gen_wind_onshore': 0, 'gen_solar': 0, 'gen_fossil_gas': 0, 'gen_hydro_pumped_storage_consumption': 0, 'humidity': 0, 'temperature': 0, 'price': 0}
Train: (26304, 20) Test: (8760, 20)
Min train date:  2015-01-01 00:00:00+01:00 Max train date:  2017-12-31 23:00:00+01:00
Min test date:  2018-01-01 00:00:00+01:00 Max test date:  2018-12-31 23:00:00+01:00


In [28]:
import time 
StartTime = time.time()

results = []

best_window_size = 168    #to be comparable with ML models

for exog_name, cols in feature_sets.items():
    print(f"{exog_name}, {cols}")
    # GRU
    m_gru, dates, y_train_full, y_predgru = run_rnn_experiment(
        df_seg=dfaux,
        feature_cols=cols,
        target=target,
        window_size=best_window_size,
        horizon=horizon,
        train_years=train_years,
        test_year=test_year,
        rnn_type="GRU"
    )
    results.append({"model":"GRU", "exogs":exog_name, **m_gru})

EndTime = time.time()
print(f"GRU Total time: {round((EndTime-StartTime)/60,1)} min")

StartTime = time.time()
for exog_name, cols in feature_sets.items():
    print(f"{exog_name}, {cols}")
    # LSTM
    m_lstm, dates, y_train_full, y_predlstm = run_rnn_experiment(
        df_seg=dfaux,
        feature_cols=cols,
        target=target,
        window_size=best_window_size,
        horizon=horizon,
        train_years=train_years,
        test_year=test_year,
        rnn_type="LSTM"
    )
    results.append({"model":"LSTM", "exogs":exog_name, **m_lstm})

EndTime = time.time()
print(f"LSTM Total time: {round((EndTime-StartTime)/60,1)} min")

#Save this data to make the ensemble
np.save("y_pred_gru2018.npy", y_predgru)
np.save("y_pred_lstm2018.npy", y_predlstm)
np.save("dates_rnn2018.npy", dates)

price+calendar+exogs, ['price', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'is_holiday', 'total_load_actual', 'gen_wind_onshore', 'gen_solar', 'gen_fossil_gas', 'gen_hydro_pumped_storage_consumption', 'humidity', 'temperature']
Epoch 1/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 27s 259ms/step - loss: 0.6028 - val_loss: 0.5855 - learning_rate: 0.0010
Epoch 2/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 25s 253ms/step - loss: 0.4074 - val_loss: 0.4768 - learning_rate: 0.0010
Epoch 3/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 25s 254ms/step - loss: 0.3677 - val_loss: 0.4619 - learning_rate: 0.0010
Epoch 4/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 25s 253ms/step - loss: 0.3469 - val_loss: 0.4376 - learning_rate: 0.0010
Epoch 5/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 25s 253ms/step - loss: 0.3366 - val_loss: 0.4212 - learning_rate: 0.0010
Epoch 6/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 25s 254ms/step - loss: 0.3229 - val_loss: 0.4433 - learning_rate: 0.0010
Epoch 7/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 25s 253ms/step - loss: 0.

In [29]:
result_df = pd.DataFrame()
result_df = pd.concat([result_df, df_somexog], ignore_index=True)      

dl_results = pd.DataFrame(results) 

# Concatenate all DataFrames into a single DataFrame
result_df = pd.concat([result_df, dl_results], ignore_index=True) 

print(result_df.sort_values(by='RMSE_mean')[['model', 'MAE_mean', 'RMSE_mean', 'MAPE_mean', 'R2_mean', 'MASE_mean']])


              model  MAE_mean  RMSE_mean  MAPE_mean  R2_mean  MASE_mean
0  LinearRegression     5.569      7.815     16.462    0.608      0.702
1  GradientBoosting     5.856      8.191     17.413    0.570      0.739
6               GRU     6.718      8.834     18.000    0.514      0.847
4          LightGBM     6.460      8.942     18.373    0.487      0.815
2      RandomForest     6.583      8.971     18.570    0.484      0.830
5           Bagging     6.621      9.015     18.670    0.479      0.835
3           XGBoost     6.610      9.127     18.657    0.466      0.834
7              LSTM     7.709     10.281     21.857    0.341      0.972


## Train GRU, LinearRegression, and GradientBoosting on 2015-2016<br> to build a Ridge/ElasticNet stacking Ensemble
### Split data: Sliding expanding-window CV

| Window | Train     | Test |
|--------|-----------|------|
| 1      | 2015–2016 | 2017 |

This is a **expanding window cross-validation**, chronologically consistent, with no look-ahead bias, and realistic for operational forecasting.<br>
An expanding-window cross-validation strategy was adopted to preserve the temporal structure of the data and avoid look-ahead bias.


In [30]:
train_years = [2015, 2016]
test_year =  2017
        
train_df = dfaux[dfaux['year'].isin(train_years)]
test_df = dfaux[dfaux['year'] == test_year]
 
# Split target  
y_train = train_df['price'].copy()               
y_test  = test_df['price'].copy()                

y_train_prev = y_train

print(f"Train: {y_train.shape} Test: {y_test.shape}")

print('Min train date: ', np.min(y_train.index), 'Max train date: ', np.max(y_train.index))
print('Min test date: ', np.min(y_test.index), 'Max test date: ', np.max(y_test.index)) 

Train: (17544,) Test: (8760,)
Min train date:  2015-01-01 00:00:00+01:00 Max train date:  2016-12-31 23:00:00+01:00
Min test date:  2017-01-01 00:00:00+01:00 Max test date:  2017-12-31 23:00:00+01:00


In [31]:
StartTime = time.time()

results = []

best_window_size = 168    #to be comparable with ML models
horizon = 24

cols = [target] + calendar_vars + exog_vars

m_gru, dates, y_train_full, y_predgru = run_rnn_experiment (df_seg=dfaux,
                                                            feature_cols=cols,
                                                            target=target,
                                                            window_size=best_window_size,
                                                            horizon=horizon,
                                                            train_years=train_years,
                                                            test_year=test_year)

EndTime = time.time()
print(f"GRU Total time: {round((EndTime-StartTime)/60,1)} min")

# Save 2017 predictions to make stacking ensemble
np.save("y_pred_gru2017.npy", y_predgru)
np.save("dates_rnn2017.npy", dates)

results.append({"model":"GRU", "exogs":"price+calendar+exogs", **m_gru})
resdlml_df = pd.DataFrame(results)

print(resdlml_df)

Epoch 1/50
65/65 ━━━━━━━━━━━━━━━━━━━━ 18s 258ms/step - loss: 0.6653 - val_loss: 0.3587 - learning_rate: 0.0010
Epoch 2/50
65/65 ━━━━━━━━━━━━━━━━━━━━ 16s 251ms/step - loss: 0.4201 - val_loss: 0.3104 - learning_rate: 0.0010
Epoch 3/50
65/65 ━━━━━━━━━━━━━━━━━━━━ 16s 252ms/step - loss: 0.3708 - val_loss: 0.2915 - learning_rate: 0.0010
Epoch 4/50
65/65 ━━━━━━━━━━━━━━━━━━━━ 16s 252ms/step - loss: 0.3519 - val_loss: 0.2968 - learning_rate: 0.0010
Epoch 5/50
65/65 ━━━━━━━━━━━━━━━━━━━━ 16s 251ms/step - loss: 0.3404 - val_loss: 0.2861 - learning_rate: 0.0010
Epoch 6/50
65/65 ━━━━━━━━━━━━━━━━━━━━ 16s 252ms/step - loss: 0.3261 - val_loss: 0.3050 - learning_rate: 0.0010
Epoch 7/50
65/65 ━━━━━━━━━━━━━━━━━━━━ 16s 251ms/step - loss: 0.3125 - val_loss: 0.2793 - learning_rate: 0.0010
Epoch 8/50
65/65 ━━━━━━━━━━━━━━━━━━━━ 16s 251ms/step - loss: 0.2995 - val_loss: 0.3039 - learning_rate: 0.0010
Epoch 9/50
65/65 ━━━━━━━━━━━━━━━━━━━━ 16s 252ms/step - loss: 0.2876 - val_loss: 0.3177 - learning_rate: 0.0010
E

In [32]:
important_exogs = ['total_load_actual',
                   'gen_wind_onshore',
                   'gen_solar',
                   'gen_fossil_gas',
                   'gen_hydro_pumped_storage_consumption',
                   'humidity',
                   'temperature']

# Create just two useful lags for each important_exogs
for col in important_exogs:
    dfaux[col + '_lag24']  = df[col].shift(24)   # same hour yesterday
    dfaux[col + '_lag168'] = df[col].shift(168)  # same hour last week

# Optional but cheap and helpful: daily average
for col in important_exogs:
    dfaux[col + '_roll24_mean'] = df[col].rolling(24).mean()
    
exog_vars = []
# Add suffixes for important_exogs
for col in important_exogs:
    exog_vars += [col + '_lag24', col + '_lag168', col + '_roll24_mean']

#Remove important_exogs
dfaux.drop(columns=important_exogs, inplace=True)

exog_vars = exog_vars + calendar_vars 

print(f"{len(exog_vars)} exogenous variables: {exog_vars}")

28 exogenous variables: ['total_load_actual_lag24', 'total_load_actual_lag168', 'total_load_actual_roll24_mean', 'gen_wind_onshore_lag24', 'gen_wind_onshore_lag168', 'gen_wind_onshore_roll24_mean', 'gen_solar_lag24', 'gen_solar_lag168', 'gen_solar_roll24_mean', 'gen_fossil_gas_lag24', 'gen_fossil_gas_lag168', 'gen_fossil_gas_roll24_mean', 'gen_hydro_pumped_storage_consumption_lag24', 'gen_hydro_pumped_storage_consumption_lag168', 'gen_hydro_pumped_storage_consumption_roll24_mean', 'humidity_lag24', 'humidity_lag168', 'humidity_roll24_mean', 'temperature_lag24', 'temperature_lag168', 'temperature_roll24_mean', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'is_holiday']


In [33]:
# Only the two best models: LinearRegression and GradientBoostingRegressor

def get_models(random_state=42):
    models = {
        "LinearRegression": LinearRegression(),
        "GradientBoosting": GradientBoostingRegressor(random_state=random_state)
    }
    return models


In [34]:
train_years = [2015, 2016]
test_year =  2017

best_window_size = 168
horizon = 24
target = "price"
        
train_df = dfaux[dfaux['year'].isin(train_years)]
test_df = dfaux[dfaux['year'] == test_year]

# y_train_full should be the original continuous series: shape (N,)
y_train_full = train_df['price']

exog_train_df = train_df[exog_vars].copy()    
exog_test_df  = test_df[exog_vars].copy()

In [35]:
# Expanding-window CV runner (direct 24-step)
StartTime = time.time()

train = slideWindow_static_exog(train_df['price'], exog_train_df, best_window_size, horizon)
test = slideWindow_static_exog(test_df['price'], exog_test_df, best_window_size, horizon)
    
X_train = train.iloc[:,:-horizon]         # All Features, except horizon columns
y_train = train.iloc[:,-horizon:]         # Target - horizon columns

X_test = test.iloc[:,:-horizon]         
y_test = test.iloc[:,-horizon:]

print(f"Shapes: {X_train.shape}, {y_train.shape}, {X_test.shape}, {y_test.shape}")

na_cols = X_train.columns[X_train.isna().any()].tolist()
print("Columns with NaN:", len(na_cols))
print(X_train[na_cols].isna().sum().sort_values(ascending=False).head(20))

# How many rows contain any NaN?
print("Rows with any NaN in X_train:", X_train.isna().any(axis=1).sum())
print("Rows with any NaN in X_test:", X_test.isna().any(axis=1).sum())

# Drop rows with NaNs in X (and also protect against NaNs in y)
mask_train = ~(X_train.isna().any(axis=1) | y_train.isna().any(axis=1))
X_train = X_train.loc[mask_train]
y_train = y_train.loc[mask_train]

mask_test = ~(X_test.isna().any(axis=1) | y_test.isna().any(axis=1))
X_test = X_test.loc[mask_test]
y_test = y_test.loc[mask_test]

print(f"X_train, y_train after cleaning: {X_train.shape}, {y_train.shape}")
print(f"X_test, y_test after cleaning: {X_test.shape}, {y_test.shape}")

df_somexog, pred_tensor_allvars2017, redmodel_names, y_trueallvars2017 = run_expanding_cv_direct_tensor(X_train, y_train, y_train_full,
                                                                                                        X_test,  y_test,
                                                                                                        'price+calendar+exogs',
                                                                                                        random_state=42)

y_test2017 =  y_test.to_numpy() if isinstance(y_test, (pd.DataFrame, pd.Series)) else np.asarray(y_test)
 
EndTime = time.time()
print(f"Total time: {round((EndTime-StartTime)/60,1)} min") 

# Concatenate all DataFrames into a single DataFrame
resdlml_df = pd.concat([resdlml_df, df_somexog], ignore_index=True)      

resdlml_df.sort_values(by=['RMSE_mean'], ascending = True, inplace=True)
resdlml_df

Shapes: (17353, 196), (17353, 24), (8569, 196), (8569, 24)
Columns with NaN: 7
exg_total_load_actual_lag168                       1
exg_gen_wind_onshore_lag168                        1
exg_gen_solar_lag168                               1
exg_gen_fossil_gas_lag168                          1
exg_gen_hydro_pumped_storage_consumption_lag168    1
exg_humidity_lag168                                1
exg_temperature_lag168                             1
dtype: int64
Rows with any NaN in X_train: 1
Rows with any NaN in X_test: 0
X_train, y_train after cleaning: (17352, 196), (17352, 24)
X_test, y_test after cleaning: (8569, 196), (8569, 24)
Total time: 2.7 min


,model,exogs,MAE_mean,RMSE_mean,MAPE_mean,R2_mean,MASE_mean
1,LinearRegression,price+calendar+exogs,5.690,8.094,13.115,0.560,0.690
2,GradientBoosting,price+calendar+exogs,6.263,8.917,13.582,0.466,0.760
0,GRU,price+calendar+exogs,7.761,10.186,16.633,0.306,0.941


In [36]:
# Save 2017 predictions to make stacking ensemble

pd.Series(redmodel_names).to_csv("redmodel_names.csv", index=False)
np.save("pred_tensor_allvars2017.npy", pred_tensor_allvars2017)
np.save("dates_ml2017.npy", test.index)
np.save("y_train_prev2015-16.npy", y_train_full)
np.save("y_test2017.npy", y_test2017)
